# Глава 4: анализ v2-результатов

Перепрогон после правок:
  - CoT v2 (audit-framing, STEP 3 micro-procedure, worked examples для clean-C)
  - Few-shot v2 (anti-examples + двухпроходный dedupe в stats_extractor)
  - Cross-verify v2 (adversarial framing + verbatim-quote requirement)
  - Stage 4 v2 (table-row matching hints + slim_test с df1/df2)
  - **Эксп. 6: table-sweep** — двухпроходное извлечение для таблиц
    (pass A: inline; pass B: per-table LLM-вызов)

Все цифры здесь — на dev-сплите v2 baseline (apples-to-apples сравнение).

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from collections import defaultdict
import sys

sys.path.insert(0, str(Path.cwd()))
from _chapter4_helpers import (
    REPO, load_run, join_gold, agg_stages, agg_by,
    fp_category_counts, fmt, fmt_pct, tex_table, write_tex,
)

DATA = REPO / "data"
RESULTS = REPO / "results"
TABLES_OUT = REPO / "notebooks" / "tables"
TABLES_OUT.mkdir(parents=True, exist_ok=True)

DEV_PATH = DATA / "dev_dataset.jsonl"
TEST_PATH = DATA / "test_dataset.jsonl"

## 1. Реестр v2-прогонов

In [ ]:
RUNS_V2 = {
    "baseline_ds":      RESULTS / "dev_baseline_v2_or_deepseek.jsonl",
    "baseline_gem":     RESULTS / "dev_baseline_v2_or_gemini.jsonl",
    "cot_ds":           RESULTS / "cot_v2_or_deepseek.jsonl",
    "cot_gem":          RESULTS / "cot_v2_or_gemini.jsonl",
    "fewshot_ds":       RESULTS / "dev_fewshot_v2_or_deepseek.jsonl",
    "fewshot_gem":      RESULTS / "dev_fewshot_v2_or_gemini.jsonl",
    "summ_ds":          RESULTS / "dev_summarised_or_deepseek.jsonl",   # v1, не перегоняли
    "summ_gem":         RESULTS / "dev_summarised_or_gemini.jsonl",     # v1, не перегоняли
    "xverify_ds2gem":   RESULTS / "dev_xverify_v2_ds_to_gem.jsonl",
    "xverify_gem2ds":   RESULTS / "dev_xverify_v2_gem_to_ds.jsonl",
    "table_sweep_ds":   RESULTS / "dev_table_sweep_or_deepseek.jsonl",
    "table_sweep_gem":  RESULTS / "dev_table_sweep_or_gemini.jsonl",
    "test_baseline_ds": RESULTS / "test_or_deepseek.jsonl",             # v1 test, ещё не перегоняли
}

# для diff'а с v1
RUNS_V1 = {
    "baseline_ds_v1":   RESULTS / "dev_baseline_or_deepseek.jsonl",
    "baseline_gem_v1":  RESULTS / "dev_baseline_or_gemini.jsonl",
    "cot_ds_v1":        RESULTS / "cot_or_deepseek.jsonl",
    "cot_gem_v1":       RESULTS / "cot_or_gemini.jsonl",
}


DATA_BY_LABEL = {}
def load_label(label, path, dev_or_test=DEV_PATH):
    rows = load_run(path)
    if path.exists() and rows:
        rows = join_gold(rows, dev_or_test)
    return rows

for label, path in {**RUNS_V2, **RUNS_V1}.items():
    if not path.exists():
        print(f"[MISSING]  {label:25s} {path.name}")
        DATA_BY_LABEL[label] = []
        continue
    ds_path = TEST_PATH if "test_" in label else DEV_PATH
    rows = load_label(label, path, ds_path)
    print(f"[ok]       {label:25s} {path.name:55s} {len(rows):4d} rows")
    DATA_BY_LABEL[label] = rows

## 2. Sanity-check v1 vs v2 baseline

Должны быть близки. Большой разрыв = подозрительно.

In [ ]:
print()
print("=== v1 → v2 baseline diff (sanity) ===")
for model in ["ds", "gem"]:
    v1 = DATA_BY_LABEL.get(f"baseline_{model}_v1") or []
    v2 = DATA_BY_LABEL.get(f"baseline_{model}") or []
    if not v1 or not v2:
        continue
    a1 = agg_stages(v1)
    a2 = agg_stages(v2)
    s3_1, s3_2 = a1["stage3"], a2["stage3"]
    s5_1, s5_2 = a1["stage5"], a2["stage5"]
    print(f"\nModel {model.upper()}:")
    print(f"  n_examples:   v1={a1['n_examples']:3d}  v2={a2['n_examples']:3d}")
    print(f"  F1:           v1={fmt(s3_1.get('f1'))}  v2={fmt(s3_2.get('f1'))}  "
          f"Δ={(s3_2.get('f1', 0) - s3_1.get('f1', 0)):+.3f}")
    print(f"  Recall:       v1={fmt(s3_1.get('recall'))}  v2={fmt(s3_2.get('recall'))}")
    print(f"  Hallucination v1={fmt_pct(s3_1.get('hallucination_rate'))}%  "
          f"v2={fmt_pct(s3_2.get('hallucination_rate'))}%")
    print(f"  Combined IDR: v1={fmt(s5_1.get('combined_idr'))}  "
          f"v2={fmt(s5_2.get('combined_idr'))}")

## 3. Helpers для генерации таблиц (как в v1, но на v2-данных)

In [ ]:
def defaultdict_groupby(rows, field):
    g = defaultdict(list)
    for r in rows:
        g[r.get(field)].append(r)
    return g


def stage3_row(label, agg):
    s3 = agg.get("stage3", {})
    return [
        label, str(agg.get("n_examples", "—")),
        str(s3.get("tp", "—")), str(s3.get("fp", "—")), str(s3.get("fn", "—")),
        fmt(s3.get("precision")), fmt(s3.get("recall")),
        fmt(s3.get("f1")), fmt(s3.get("field_accuracy")),
        fmt_pct(s3.get("hallucination_rate")),
    ]


def stage5_row(label, agg):
    s5 = agg.get("stage5", {})
    return [
        label, str(agg.get("n_examples", "—")),
        str(s5.get("tpv", "—")), str(s5.get("fpv", "—")), str(s5.get("fnv", "—")),
        fmt(s5.get("inconsistency_detection_rate")),
        fmt(s5.get("false_alarm_rate")),
        fmt(s5.get("combined_idr")),
        fmt(s5.get("combined_far")),
    ]


def compact_summary(label, rows):
    a = agg_stages(rows)
    s3, s4, s5 = a["stage3"], a["stage4"], a["stage5"]
    return [
        label, str(a["n_examples"]),
        fmt(s3.get("f1")),
        fmt(s4.get("primary_direction_accuracy")),
        fmt(s5.get("combined_idr")),
        fmt_pct(s3.get("hallucination_rate")),
    ]

## 4. Таблица 4.1 — Baseline v2

In [ ]:
agg_baseline = {
    "DeepSeek-V3 (dev v2)":   agg_stages(DATA_BY_LABEL["baseline_ds"]),
    "Gemini 2.5 Flash (dev v2)": agg_stages(DATA_BY_LABEL["baseline_gem"]),
}
if DATA_BY_LABEL["test_baseline_ds"]:
    agg_baseline["DeepSeek-V3 (test v1)"] = agg_stages(DATA_BY_LABEL["test_baseline_ds"])

write_tex(TABLES_OUT / "table_4_1_baseline_stage3.tex", tex_table(
    label="tab:exp_baseline_stage3",
    caption=("Baseline (v2 promptом): метрики извлечения тестов на этапе~3."),
    headers=["Прогон", "n", "TP", "FP", "FN", "Precision", "Recall",
             "$F_1$", "Field Acc", "Hall., \\%"],
    rows=[stage3_row(k, v) for k, v in agg_baseline.items()],
))

write_tex(TABLES_OUT / "table_4_1_baseline_stage5.tex", tex_table(
    label="tab:exp_baseline_stage5",
    caption="Baseline v2: метрики самосогласованности на этапе~5.",
    headers=["Прогон", "n", "TP$_v$", "FP$_v$", "FN$_v$",
             "IDR (p)", "FAR (p)", "comb.~IDR", "comb.~FAR"],
    rows=[stage5_row(k, v) for k, v in agg_baseline.items()],
))
print("\n=== Таблица 4.1 (baseline v2) ===")
for k, v in agg_baseline.items():
    s3 = v["stage3"]
    print(f"  {k}: F1={fmt(s3.get('f1'))}, Recall={fmt(s3.get('recall'))}, "
          f"Hall={fmt_pct(s3.get('hallucination_rate'))}%")

### Baseline по environment (для гипотез H1, H2)

In [ ]:
def env_table_rows():
    out = []
    for env in ["apa", "non_apa", "text", "table", "two_apa", "two_text", "no_test"]:
        ds_rows = [r for r in DATA_BY_LABEL["baseline_ds"] if r.get("environment") == env]
        gem_rows = [r for r in DATA_BY_LABEL["baseline_gem"] if r.get("environment") == env]
        if not ds_rows and not gem_rows:
            continue
        ds_a = agg_stages(ds_rows)
        gem_a = agg_stages(gem_rows)
        out.append([
            env, str(max(ds_a["n_examples"], gem_a["n_examples"])),
            fmt(ds_a["stage3"].get("recall")),
            fmt(ds_a["stage3"].get("f1")),
            fmt_pct(ds_a["stage3"].get("hallucination_rate")),
            fmt(gem_a["stage3"].get("recall")),
            fmt(gem_a["stage3"].get("f1")),
            fmt_pct(gem_a["stage3"].get("hallucination_rate")),
        ])
    return out

write_tex(TABLES_OUT / "table_4_1_baseline_env.tex", tex_table(
    label="tab:exp_baseline_env",
    caption=("Baseline v2: разрез по~\\texttt{environment}."),
    headers=["env", "n", "Rec.\\,DS", "$F_1$\\,DS", "Hall.\\,DS",
             "Rec.\\,Gem", "$F_1$\\,Gem", "Hall.\\,Gem"],
    rows=env_table_rows(),
))

## 5. Таблица 4.3 — CoT v2 (главный фикс: clean-C класс)

In [ ]:
def cot_summary_row(label, rows):
    a = agg_stages(rows)
    s4 = a["stage4"]
    s5 = a["stage5"]
    cc = [r for r in rows if r.get("error_type") == "wrong_conclusion_clean"]
    a_cc = agg_stages(cc) if cc else None
    s5_cc = (a_cc["stage5"] if a_cc else {})
    return [
        label,
        fmt(s4.get("primary_direction_accuracy")),
        fmt(s5.get("inconsistency_detection_rate")),
        fmt(s5.get("combined_idr")),
        fmt(s5_cc.get("combined_idr")) if a_cc else "—",
    ]

cot_rows = [
    cot_summary_row("DS, baseline",  DATA_BY_LABEL["baseline_ds"]),
    cot_summary_row("DS, + CoT v2",  DATA_BY_LABEL["cot_ds"]),
    cot_summary_row("Gem, baseline", DATA_BY_LABEL["baseline_gem"]),
    cot_summary_row("Gem, + CoT v2", DATA_BY_LABEL["cot_gem"]),
]

write_tex(TABLES_OUT / "table_4_3_cot.tex", tex_table(
    label="tab:exp_cot",
    caption=("Эксперимент~2 (v2): Chain-of-Thought с audit-framing и~"
             "worked examples для~clean-C. Колонка <<Combined~IDR (clean-C)>>"
             "~--- на~30~примерах класса \\texttt{wrong\\_conclusion\\_clean}."),
    headers=["Прогон", "Primary Acc", "IDR (p)", "Combined IDR",
             "Combined IDR (clean-C)"],
    rows=cot_rows,
))

print("\n=== Таблица 4.3 (CoT v2) ===")
for r in cot_rows:
    print(f"  {r[0]:18s}: PrimaryAcc={r[1]}  IDR(p)={r[2]}  Comb.IDR={r[3]}  clean-C={r[4]}")


# IDR по error_type для CoT
def idr_by_error_rows(label, rows):
    out = []
    by_err = defaultdict_groupby(rows, "error_type")
    for et in ["wrong_conclusion_clean", "wrong_conclusion", "wrong_pvalue",
               "rounding", "transcription"]:
        rrs = by_err.get(et, [])
        if not rrs:
            continue
        a = agg_stages(rrs)
        s5 = a["stage5"]
        out.append([
            label, et, str(len(rrs)),
            fmt(s5.get("inconsistency_detection_rate")),
            fmt(s5.get("combined_idr")),
        ])
    return out

idr_breakdown = []
for label, key in [("DS, baseline", "baseline_ds"),
                    ("DS, + CoT v2", "cot_ds"),
                    ("Gem, baseline", "baseline_gem"),
                    ("Gem, + CoT v2", "cot_gem")]:
    idr_breakdown.extend(idr_by_error_rows(label, DATA_BY_LABEL[key]))

write_tex(TABLES_OUT / "table_4_3_cot_by_error.tex", tex_table(
    label="tab:exp_cot_by_error",
    caption="Эксперимент~2 (v2): Combined IDR по~классам ошибок.",
    headers=["Прогон", "error\\_type", "n", "IDR (p)", "Combined IDR"],
    rows=idr_breakdown,
))

## 6. Таблица 4.4 — Cross-verify v2

In [ ]:
def xverify_diagnostics(rows):
    keys = ["n_proposed", "n_verified", "n_corrected", "n_removed",
            "n_added", "n_unmentioned", "n_final"]
    sums = {k: 0 for k in keys}
    for r in rows:
        diag = r.get("cross_verify_diagnostics") or {}
        for k in keys:
            sums[k] += diag.get(k, 0) or 0
    return sums


xv_rows = []
for label, key in [("DS, baseline",  "baseline_ds"),
                    ("Gem, baseline", "baseline_gem"),
                    ("DS $\\to$ Gem v2",  "xverify_ds2gem"),
                    ("Gem $\\to$ DS v2",  "xverify_gem2ds")]:
    a = agg_stages(DATA_BY_LABEL[key])
    s3, s5 = a["stage3"], a["stage5"]
    xv_rows.append([
        label, str(a["n_examples"]),
        str(s3.get("tp", "—")), str(s3.get("fp", "—")), str(s3.get("fn", "—")),
        fmt(s3.get("precision")), fmt(s3.get("recall")),
        fmt(s3.get("f1")), fmt(s5.get("combined_idr")),
    ])

write_tex(TABLES_OUT / "table_4_4_xverify.tex", tex_table(
    label="tab:exp_xverify",
    caption=("Эксперимент~3 (v2): cross-verify с~adversarial framing и~"
             "verbatim-quote requirement."),
    headers=["Прогон", "n", "TP", "FP", "FN", "Precision", "Recall",
             "$F_1$", "Comb.~IDR"],
    rows=xv_rows,
))

diag_rows = []
for label, key in [("DS $\\to$ Gem v2", "xverify_ds2gem"),
                    ("Gem $\\to$ DS v2", "xverify_gem2ds")]:
    d = xverify_diagnostics(DATA_BY_LABEL[key])
    diag_rows.append([
        label,
        str(d["n_proposed"]), str(d["n_verified"]), str(d["n_corrected"]),
        str(d["n_removed"]), str(d["n_added"]), str(d["n_final"]),
    ])

write_tex(TABLES_OUT / "table_4_4_xverify_diag.tex", tex_table(
    label="tab:exp_xverify_diag",
    caption=("Cross-verify v2: счётчики действий verifier'а. "
             "Ожидание после промпт-фикса: \\texttt{n\\_removed} вырастет, "
             "\\texttt{n\\_added} упадёт (модель станет precision-фильтром)."),
    headers=["Прогон", "n\\_proposed", "n\\_verified", "n\\_corrected",
             "n\\_removed", "n\\_added", "n\\_final"],
    rows=diag_rows,
))

print("\n=== Таблица 4.4 (xverify v2) ===")
for r in xv_rows:
    print(f"  {r[0]:20s}: F1={r[7]}  Comb.IDR={r[8]}")
for r in diag_rows:
    print(f"  {r[0]:20s}: removed={r[4]} added={r[5]} corrected={r[3]}")

## 7. Таблица 4.5 — Summarisation (без изменений — v1 cifry)

In [ ]:
def summ_stats(rows):
    n_used = 0
    n_calls_total = 0
    ratios = []
    for r in rows:
        s = r.get("summarization_stats") or {}
        if s.get("n_calls", 0) > 0:
            n_used += 1
            n_calls_total += s["n_calls"]
            if s.get("ratio") is not None:
                ratios.append(s["ratio"])
    return {
        "n_used": n_used,
        "n_calls_total": n_calls_total,
        "ratio_mean": (sum(ratios) / len(ratios)) if ratios else None,
    }


summ_rows = []
for model_label, base_key, summ_key in [
    ("DeepSeek-V3", "baseline_ds", "summ_ds"),
    ("Gemini 2.5 Flash", "baseline_gem", "summ_gem"),
]:
    for src_label, src_filter in [("synthetic", "synthetic"), ("real", "real"), ("all", None)]:
        base_rows = [r for r in DATA_BY_LABEL[base_key]
                     if not src_filter or r.get("source") == src_filter]
        summ_rows_data = [r for r in DATA_BY_LABEL[summ_key]
                          if not src_filter or r.get("source") == src_filter]
        a_base = agg_stages(base_rows)
        a_summ = agg_stages(summ_rows_data)
        ss = summ_stats(summ_rows_data)
        s3_b, s3_s = a_base["stage3"], a_summ["stage3"]
        summ_rows.append([
            f"{model_label}, {src_label}",
            str(a_base["n_examples"]),
            fmt(s3_b.get("recall")),       fmt(s3_s.get("recall")),
            fmt(s3_b.get("f1")),           fmt(s3_s.get("f1")),
            fmt_pct(s3_b.get("hallucination_rate")),
            fmt_pct(s3_s.get("hallucination_rate")),
            fmt(ss["ratio_mean"]),
            str(ss["n_used"]),
        ])

write_tex(TABLES_OUT / "table_4_5_summarisation.tex", tex_table(
    label="tab:exp_summarisation",
    caption=("Эксперимент~4: вклад суммаризации на~этапе~2 (v1 цифры, "
             "summarisation не~перегоняли)."),
    headers=["Прогон / src", "n", "Recall (b)", "Recall (s)",
             "$F_1$ (b)", "$F_1$ (s)", "Hall., \\% (b)", "Hall., \\% (s)",
             "ratio", "n used"],
    rows=summ_rows,
))

## 8. Таблица 4.6 — Few-shot v2

In [ ]:
fewshot_rows = []
for model_label, base_key, fs_key in [
    ("DeepSeek-V3", "baseline_ds", "fewshot_ds"),
    ("Gemini 2.5 Flash", "baseline_gem", "fewshot_gem"),
]:
    for env_label, env_filter in [("all", None), ("table", "table"), ("real", None)]:
        if env_label == "real":
            base_rows = [r for r in DATA_BY_LABEL[base_key] if r.get("source") == "real"]
            fs_rows = [r for r in DATA_BY_LABEL[fs_key] if r.get("source") == "real"]
        else:
            base_rows = DATA_BY_LABEL[base_key]
            fs_rows = DATA_BY_LABEL[fs_key]
            if env_filter:
                base_rows = [r for r in base_rows if r.get("environment") == env_filter]
                fs_rows = [r for r in fs_rows if r.get("environment") == env_filter]
        a_b = agg_stages(base_rows)
        a_f = agg_stages(fs_rows)
        s3_b, s3_f = a_b["stage3"], a_f["stage3"]
        fp_b = fp_category_counts(base_rows)
        fp_f = fp_category_counts(fs_rows)
        fewshot_rows.append([
            f"{model_label}, {env_label}",
            str(a_b["n_examples"]),
            fmt(s3_b.get("recall")), fmt(s3_f.get("recall")),
            fmt(s3_b.get("f1")),     fmt(s3_f.get("f1")),
            str(fp_b.get("duplicate", 0)),
            str(fp_f.get("duplicate", 0)),
        ])

write_tex(TABLES_OUT / "table_4_6_fewshot.tex", tex_table(
    label="tab:exp_fewshot",
    caption=("Эксперимент~5 (v2): few-shot для~таблиц с~anti-examples и~"
             "двухпроходным dedupe."),
    headers=["Прогон / срез", "n", "Recall (b)", "Recall (f)",
             "$F_1$ (b)", "$F_1$ (f)",
             "duplicate (b)", "duplicate (f)"],
    rows=fewshot_rows,
))

## 9. Таблица 4.8 — НОВЫЙ Эксп. 6: table-sweep

Двухпроходное извлечение (pass A inline + pass B per-table).
Прямой ответ на коммент научника №3.

In [ ]:
def table_sweep_diagnostics(rows):
    n_tables_total = sum(r.get("n_tables_detected", 0) or 0 for r in rows)
    n_inline_total = sum(r.get("n_inline_predicted", 0) or 0 for r in rows)
    n_table_total = sum(r.get("n_table_predicted", 0) or 0 for r in rows)
    return {
        "n_tables_total": n_tables_total,
        "n_inline_total": n_inline_total,
        "n_table_total": n_table_total,
    }


ts_rows = []
for model_label, base_key, ts_key in [
    ("DeepSeek-V3", "baseline_ds", "table_sweep_ds"),
    ("Gemini 2.5 Flash", "baseline_gem", "table_sweep_gem"),
]:
    for src_label, src_filter in [("synthetic", "synthetic"),
                                   ("real", "real"),
                                   ("table-env only", None),
                                   ("all", None)]:
        if src_label == "table-env only":
            base_rows = [r for r in DATA_BY_LABEL[base_key]
                         if r.get("environment") == "table"]
            ts_rows_data = [r for r in DATA_BY_LABEL[ts_key]
                            if r.get("environment") == "table"]
        elif src_filter:
            base_rows = [r for r in DATA_BY_LABEL[base_key] if r.get("source") == src_filter]
            ts_rows_data = [r for r in DATA_BY_LABEL[ts_key] if r.get("source") == src_filter]
        else:
            base_rows = DATA_BY_LABEL[base_key]
            ts_rows_data = DATA_BY_LABEL[ts_key]
        a_b = agg_stages(base_rows)
        a_t = agg_stages(ts_rows_data)
        s3_b, s3_t = a_b["stage3"], a_t["stage3"]
        ts_rows.append([
            f"{model_label}, {src_label}",
            str(a_t["n_examples"]),
            fmt(s3_b.get("recall")), fmt(s3_t.get("recall")),
            fmt(s3_b.get("f1")),     fmt(s3_t.get("f1")),
            fmt_pct(s3_b.get("hallucination_rate")),
            fmt_pct(s3_t.get("hallucination_rate")),
        ])

write_tex(TABLES_OUT / "table_4_8_table_sweep.tex", tex_table(
    label="tab:exp_table_sweep",
    caption=("Эксперимент~6: двухпроходное извлечение~--- pass~A inline без~"
             "markdown-таблиц~+ pass~B по~отдельному LLM-вызову на~каждую "
             "таблицу с~table-focused промптом. Колонки <<b>>~--- baseline v2, "
             "<<t>>~--- table-sweep."),
    headers=["Прогон / срез", "n", "Recall (b)", "Recall (t)",
             "$F_1$ (b)", "$F_1$ (t)",
             "Hall., \\% (b)", "Hall., \\% (t)"],
    rows=ts_rows,
))

# diagnostics — сколько таблиц задетектили, разделение inline vs table
ts_diag_rows = []
for model_label, ts_key in [("DeepSeek-V3", "table_sweep_ds"),
                              ("Gemini 2.5 Flash", "table_sweep_gem")]:
    for src_label, src_filter in [("synthetic", "synthetic"), ("real", "real"), ("all", None)]:
        if src_filter:
            rows = [r for r in DATA_BY_LABEL[ts_key] if r.get("source") == src_filter]
        else:
            rows = DATA_BY_LABEL[ts_key]
        d = table_sweep_diagnostics(rows)
        ts_diag_rows.append([
            f"{model_label}, {src_label}",
            str(len(rows)),
            str(d["n_tables_total"]),
            str(d["n_inline_total"]),
            str(d["n_table_total"]),
        ])

write_tex(TABLES_OUT / "table_4_8_table_sweep_diag.tex", tex_table(
    label="tab:exp_table_sweep_diag",
    caption=("Table-sweep diagnostics: число задетектированных таблиц и~"
             "распределение предсказанных тестов между inline-проходом и~"
             "per-table проходом."),
    headers=["Прогон / срез", "n примеров", "n таблиц",
             "preds inline", "preds tables"],
    rows=ts_diag_rows,
))

print("\n=== Таблица 4.8 (table-sweep, новый эксп. 6) ===")
for r in ts_rows:
    print(f"  {r[0]:38s}: F1 {r[4]} → {r[5]}  Recall {r[2]} → {r[3]}")

## 10. Таблица 4.7 — Сравнение моделей и стратегий (сводная)

In [ ]:
model_compare_rows = [
    compact_summary("DS, baseline v2",       DATA_BY_LABEL["baseline_ds"]),
    compact_summary("DS, + CoT v2",          DATA_BY_LABEL["cot_ds"]),
    compact_summary("DS, + few-shot v2",     DATA_BY_LABEL["fewshot_ds"]),
    compact_summary("DS, + summarised",      DATA_BY_LABEL["summ_ds"]),
    compact_summary("DS, + table-sweep",     DATA_BY_LABEL["table_sweep_ds"]),
    compact_summary("DS $\\to$ Gem v2",      DATA_BY_LABEL["xverify_ds2gem"]),
    compact_summary("Gem, baseline v2",      DATA_BY_LABEL["baseline_gem"]),
    compact_summary("Gem, + CoT v2",         DATA_BY_LABEL["cot_gem"]),
    compact_summary("Gem, + few-shot v2",    DATA_BY_LABEL["fewshot_gem"]),
    compact_summary("Gem, + summarised",     DATA_BY_LABEL["summ_gem"]),
    compact_summary("Gem, + table-sweep",    DATA_BY_LABEL["table_sweep_gem"]),
    compact_summary("Gem $\\to$ DS v2",      DATA_BY_LABEL["xverify_gem2ds"]),
]

write_tex(TABLES_OUT / "table_4_7_models.tex", tex_table(
    label="tab:exp_models",
    caption=("Сравнение конфигураций (v2, dev). Headline-метрики: $F_1$, "
             "Primary Acc, Combined IDR, Hallucination Rate."),
    headers=["Прогон", "n", "$F_1$", "Primary Acc", "Comb.~IDR", "Hall., \\%"],
    rows=model_compare_rows,
))

## 11. Анализ ошибок (FP-категории + IDR/FAR + F1 по env)

In [ ]:
def fp_dist_row(label, rows):
    fp = fp_category_counts(rows)
    total = sum(fp.values())
    keys = ["duplicate", "wrong_test_type", "off_stat_value", "complete_fabrication"]
    return [label, str(total)] + [
        f"{fp.get(k, 0)} ({fmt_pct(fp.get(k, 0) / total) if total else '—'}\\%)"
        for k in keys
    ]

fp_rows = [
    fp_dist_row("DS, baseline v2",  DATA_BY_LABEL["baseline_ds"]),
    fp_dist_row("Gem, baseline v2", DATA_BY_LABEL["baseline_gem"]),
    fp_dist_row("DS, + CoT v2",     DATA_BY_LABEL["cot_ds"]),
    fp_dist_row("Gem, + CoT v2",    DATA_BY_LABEL["cot_gem"]),
    fp_dist_row("DS, + few-shot v2", DATA_BY_LABEL["fewshot_ds"]),
    fp_dist_row("Gem, + few-shot v2", DATA_BY_LABEL["fewshot_gem"]),
    fp_dist_row("DS, + table-sweep", DATA_BY_LABEL["table_sweep_ds"]),
    fp_dist_row("Gem, + table-sweep", DATA_BY_LABEL["table_sweep_gem"]),
]

write_tex(TABLES_OUT / "table_4_9_fp_categories.tex", tex_table(
    label="tab:exp_fp_categories",
    caption="Распределение FP по~категориям \\texttt{classify\\_fp}.",
    headers=["Прогон", "FP всего", "duplicate", "wrong\\_type",
             "off\\_value", "fabrication"],
    rows=fp_rows,
))

## 12. Сводный markdown-отчёт

In [ ]:
def summary_md():
    out = ["# Глава 4 — v2-сводка результатов\n",
           "Сгенерировано `notebooks/chapter4_analysis_v2.py`.\n"]
    out.append("\n## Baseline v2 (dev)\n")
    for label, key in [("DeepSeek-V3", "baseline_ds"), ("Gemini 2.5 Flash", "baseline_gem")]:
        a = agg_stages(DATA_BY_LABEL[key])
        s3, s5 = a["stage3"], a["stage5"]
        out.append(
            f"- **{label}**: n={a['n_examples']}, "
            f"P={fmt(s3.get('precision'))}, "
            f"R={fmt(s3.get('recall'))}, "
            f"F1={fmt(s3.get('f1'))}, "
            f"Hall={fmt_pct(s3.get('hallucination_rate'))}%, "
            f"IDR={fmt(s5.get('inconsistency_detection_rate'))}, "
            f"Comb.IDR={fmt(s5.get('combined_idr'))}"
        )

    out.append("\n## CoT v2 — главный фикс: clean-C класс\n")
    for label, key in [("DS, baseline", "baseline_ds"), ("DS, + CoT v2", "cot_ds"),
                       ("Gem, baseline", "baseline_gem"), ("Gem, + CoT v2", "cot_gem")]:
        rows = DATA_BY_LABEL[key]
        cc = [r for r in rows if r.get("error_type") == "wrong_conclusion_clean"]
        if not cc:
            continue
        a = agg_stages(cc)
        out.append(f"- **{label}** clean-C (n={len(cc)}): Comb.IDR={fmt(a['stage5'].get('combined_idr'))}")

    out.append("\n## Cross-verify v2 — adversarial framing\n")
    for label, key in [("DS $\\to$ Gem v2", "xverify_ds2gem"), ("Gem $\\to$ DS v2", "xverify_gem2ds")]:
        a = agg_stages(DATA_BY_LABEL[key])
        d = xverify_diagnostics(DATA_BY_LABEL[key])
        out.append(
            f"- **{label}**: F1={fmt(a['stage3'].get('f1'))}, "
            f"removed={d['n_removed']}, added={d['n_added']}, corrected={d['n_corrected']}"
        )

    out.append("\n## Эксп. 6 — Table-sweep (новый)\n")
    for label, key in [("DS, baseline", "baseline_ds"), ("DS, + table-sweep", "table_sweep_ds"),
                       ("Gem, baseline", "baseline_gem"), ("Gem, + table-sweep", "table_sweep_gem")]:
        rows = [r for r in DATA_BY_LABEL[key] if r.get("source") == "real"]
        if not rows:
            continue
        a = agg_stages(rows)
        s3 = a["stage3"]
        out.append(f"- **{label}** real (n={a['n_examples']}): "
                   f"Recall={fmt(s3.get('recall'))}, F1={fmt(s3.get('f1'))}")

    out.append("\n## Все таблицы\n")
    for tex in sorted(TABLES_OUT.glob("table_4_*.tex")):
        out.append(f"- `{tex.relative_to(REPO)}`")
    return "\n".join(out) + "\n"


md_path = TABLES_OUT / "chapter4_summary.md"
md_path.write_text(summary_md(), encoding="utf-8")
print(f"\n=== Summary → {md_path} ===\n")
print(md_path.read_text())